In [6]:
import sys, os
from pathlib import Path
work_dir = os.getcwd()
proj_dir = str(Path(work_dir).resolve().parents[0])
sys.path.append(proj_dir)
import time
import sympy as sp
import torch

from bc_learn.Config import Config
from bc_learn.Draw import Draw
from bc_learn.Finetuning import Finetuner
from bc_learn.Generate_data import Data
# from bc_learn.Net_pre import Net, Learner
from bc_learn.Net import Learner
from verify.kvh_verify import KVH


def print_success():
    heart = """
          *****         *****
       ***********   *************
     *************** ***************
    *********************************
     *******************************
      ****** SUCCESS SUCCESS ******
       ***************************
         ************************
           *********************
             *****************
               *************
                 *********
                   *****
                     *
    """

    print(heart)

In [7]:
import numpy as np

seed = 2025
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
np.random.seed(seed)

In [8]:
from rl_train.Examples import get_example_by_id, get_example_by_name, Example, Zones
class F:
    @staticmethod
    def f1(x): #(1 / (1 + np.sin(x)**2)) 0.00162777504080
        return 0.0768 *x**4  - 0.3808 *x**2  + 0.9280
    @staticmethod
    def f2(x): #(np.sin(x) / (1 + np.sin(x)**2))0.00378772829829
        return -0.1060 *x**3  + 0.6139 *x
    @staticmethod
    def f3(x): #(np.sin(x) * np.cos(x) / (1 + np.sin(x)**2))0.00140085561943
        return 0.0663 *x**5  - 0.4553 *x**3  + 0.6998 *x
    @staticmethod
    def f4(x): #(np.cos(x) / (1 + np.sin(x)**2))0.00342303163390
        return 0.0915 *x**4  - 0.6153 *x**2  + 0.8964
train_ex = Example(
        n_obs=3,
        u_dim=1,
        D_zones=Zones('box', low=[-2.2]*3, up=[2.2]*3),
        I_zones=Zones('box', low=[-0.2]*3, up=[0.2]*3),
        U_zones=Zones('box', low=[1]*3, up=[2]*3),  
        f=[
            lambda x, u: 30*(-0.1576*x[0]**3  + 0.9981*x[0]) + 15*u[0]*(-4.99998744e-01*x[0]**2+4.16558586e-02*x[0]**4-1.35953076e-03*x[0]**6+0.99999998),  # ẋ₁ = 30sin(x₁) + 15ũcos(x₁)
            lambda x, u: -20*(-4.99998744e-01*x[2]**2+4.16558586e-02*x[2]**4-1.35953076e-03*x[2]**6+0.99999998)*(-0.1576*x[2]**3  + 0.9981*x[2]) + u[0]*(-4.99998744e-01*x[2]**2+4.16558586e-02*x[2]**4-1.35953076e-03*x[2]**6+0.99999998)**2,  # ẋ₂ = -20cos(x₃)sin(x₃) + ũcos²(x₃)
            lambda x, u: x[1]  # ẋ₃ = x₂ (assuming third state is integrated from second)
        ],
        u=3.0,  # Control input bound (assuming normalized ũ ∈ [-1,1])
        dense=4,
        units=64,
        dt=0.01,
        max_episode=1500,
        goal='avoid',  # Control objective (avoid unsafe set)
        name='nonpoly3-bicycle-steering'
    )

# 转换例子

eps = 0.0122
kvh_ex = Example(
        n_obs=3 + 1,
        u_dim=1,
        D_zones=Zones('box', low=[-2.2]*3 + [-eps], up=[2.2]*3 + [-eps]),
        I_zones=Zones('box', low=[-0.2]*3 + [-eps], up=[0.2]*3 + [-eps]),
        U_zones=Zones('box', low=[1]*3 + [-eps], up=[2]*3 + [-eps]),  
        f=[
            lambda x, u: 30*(-0.1576*x[0]**3  + 0.9981*x[0]) + 15*u[0]*(-4.99998744e-01*x[0]**2+4.16558586e-02*x[0]**4-1.35953076e-03*x[0]**6+0.99999998),  # ẋ₁ = 30sin(x₁) + 15ũcos(x₁)
            lambda x, u: -20*(-4.99998744e-01*x[2]**2+4.16558586e-02*x[2]**4-1.35953076e-03*x[2]**6+0.99999998)*(-0.1576*x[2]**3  + 0.9981*x[2]) + u[0]*(-4.99998744e-01*x[2]**2+4.16558586e-02*x[2]**4-1.35953076e-03*x[2]**6+0.99999998)**2,  # ẋ₂ = -20cos(x₃)sin(x₃) + ũcos²(x₃)
            lambda x, u: x[1],  # ẋ₃ = x₂ (assuming third state is integrated from second)
            lambda x, u: 0.0,
        ],
        u=3.0,  # Control input bound (assuming normalized ũ ∈ [-1,1])
        dense=4,
        units=64,
        dt=0.01,
        max_episode=1500,
        goal='avoid',  # Control objective (avoid unsafe set)
        name='nonpoly3-bicycle-steering'
    )

with open(f'{proj_dir}/controller/{train_ex.name}.txt', 'r', encoding='utf-8') as f:
    controller = f.readline()
print(controller)

0.111012027023896*x1**2 + 0.0115653758949475*x1*x2 + 0.0636512079496508*x1 + 14.2304986855668*x2**2 - 0.430668456489666*x2


In [9]:
opts = {
    'example': train_ex,
    'lr': 0.04,
    'batch_size': 500,
    'margin': 1.5,
    'hidden_neurons': [10],
    'activation': ['SKIP']
}
config = Config(**opts)

epoch = 5
l = 12
config_fine = (100, 14, 0.1, 500, 1)
adaptive_margin=False

learner = Learner(config)
opt = torch.optim.AdamW(learner.net.parameters(), lr=config.lr)
data = Data(config, controller)


In [10]:

kvh = KVH(kvh_ex, kvh_ex.n_obs, l)
kvh.pos, vis = 1, False

t_learn, t_kvh, t_finetune = 0, 0, 0

for i in range(epoch):
    if adaptive_margin:
        config.margin = config.margin * 2
    print(f'Controller for epoch {i + 1}:{controller}')
    init, unsafe, domain, domain_dot = data.generate_data()
    t1 = time.time()
    print(f'---------------------------------------\nStart training--epoch {i + 1}\n'
            '---------------------------------------')
    learner.learn(opt, (init, unsafe, domain), domain_dot)

    t2 = time.time()
    t_learn += t2 - t1

    bc = learner.net.get_barrier()
    # print('bc:', bc)
    multiplier = learner.net.get_mul()
    # print('multiplier', multiplier)
    kvh.update_barrier(bc, multiplier, sp.sympify(controller))

    t3 = time.time()

    state = kvh.verify_all()

    t4 = time.time()
    t_kvh += t4 - t3
    if state:
        vis = True
        print_success()
        print(f'In the {i + 1} epoch, barrier certificate verification successful!')
        print(f'Barrier certificate:{bc}')
        print(f'Controller:{controller}')
        print(f'The time of learn:{t_learn}s')
        print(f'The time of verify:{t_kvh}s')
        print(f'The time of finetune:{t_finetune}s')
        if config.example.n_obs == 2:
            draw = Draw(config.example, bc, controller)
            draw.draw()
        break

    if vis:
        break

    print(f'Epoch {i + 1} failed verification barrier certificate:{bc}')
    print(f'---------------------------------------\nStart fine tuning--epoch {i + 1}\n'
            '---------------------------------------')

    finetuner = Finetuner(bc, config.example, controller, kvh, multiplier, config.device)

    t5 = time.time()

    finetuner.learn(*config_fine)

    t6 = time.time()
    t_finetune += t6 - t5
    controller, multiplier = finetuner.net.get_controller(config.example.n_obs)
    # print('controller:', controller)
    kvh.update_barrier(bc, multiplier, sp.sympify(controller))

    t3 = time.time()

    result = kvh.verify_all()

    t4 = time.time()
    t_kvh += t4 - t3
    if result:
        vis = True
        print_success()
        print(f'In the {i + 1} epoch, after the fine-tuning, barrier certificate verification successful!')
        print(f'Barrier certificate:{bc}')
        print(f'Controller:{controller}')
        print(f'The time of learn:{t_learn}s')
        print(f'The time of verify:{t_kvh}s')
        print(f'The time of finetune:{t_finetune}s')
        if config.example.n_obs == 2:
            draw = Draw(config.example, bc, controller)
            draw.draw()
        break
    else:
        data.update(controller)

if not vis:
    print('No barrier certificate found!')


Initialization completed!
Controller for epoch 1:0.111012027023896*x1**2 + 0.0115653758949475*x1*x2 + 0.0636512079496508*x1 + 14.2304986855668*x2**2 - 0.430668456489666*x2
---------------------------------------
Start training--epoch 1
---------------------------------------
Init samples: 500 Unsafe samples: 500 Lie samples 500
10 - loss: 6.417898178100586 - accuracy init: 0.0 accuracy unsafe: 0.0 - accuracy Lie: 25.2
20 - loss: 4.965494155883789 - accuracy init: 0.0 accuracy unsafe: 0.0 - accuracy Lie: 19.0
30 - loss: 4.281680583953857 - accuracy init: 0.0 accuracy unsafe: 0.0 - accuracy Lie: 30.4
40 - loss: 3.7673282623291016 - accuracy init: 0.0 accuracy unsafe: 0.0 - accuracy Lie: 34.6
50 - loss: 3.65688419342041 - accuracy init: 0.0 accuracy unsafe: 0.0 - accuracy Lie: 42.8
60 - loss: 3.260800361633301 - accuracy init: 0.0 accuracy unsafe: 0.0 - accuracy Lie: 50.4
70 - loss: 3.2246508598327637 - accuracy init: 0.0 accuracy unsafe: 56.0 - accuracy Lie: 53.0
80 - loss: 2.52907061576

KeyError: 'x1**3*x3**12'